# Lab 07 · Hybrid MPI + OpenMP · one rank per socket, threads within

MPI across nodes handles distribution; OpenMP within each rank exploits shared memory. The hybrid combination usually beats pure MPI on modern many-core nodes because it (a) reduces the total number of MPI ranks and thus the amount of halo data, and (b) can share caches within a socket.

**Prerequisites.** Lab 04 (OpenMP pitfalls), Lab 06 (MPI heat).

**Builds toward.** Labs 08-10 replace the OpenMP layer with a GPU offload.

> **📚 Where to look when you're stuck**
>
> - [**MPI + OpenMP thread support**](https://www.mpich.org/static/docs/latest/www3/MPI_Init_thread.html)
> - [**Best practices for hybrid MPI+OpenMP**](https://docs.nersc.gov/jobs/examples/) (NERSC)



## How this notebook works

Same three surfaces as lab 01 and 02: **[Hub]**, **[Hub -> Crux]**, **[Crux compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab07", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab07 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
    check("lab06 MPI binary", remoteFileExists(env['HPC_LAB_DIR'].replace('lab07','lab06') + '/heat2Dmpi')),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> Crux] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab07 dir ready')


## Part 1 · The hybrid source · one pragma added

Take `heat2Dmpi.c` from lab 06 and add `#pragma omp parallel for` above the stencil update loop, exactly like lab 03. Call `MPI_Init_thread` with `MPI_THREAD_FUNNELED` at the top — you're telling the MPI runtime that only the master thread will make MPI calls (typical for hybrid stencil codes).


In [ ]:
# [Hub -> Crux] Fetch, patch, ship.
sshGet(env['HPC_LAB_DIR'].replace('lab07','lab06') + '/heat2Dmpi.c',
       str(labDir/'heat2Dhyb.c'))
src = (labDir/'heat2Dhyb.c').read_text()
if '#include <omp.h>' not in src:
    src = src.replace('#include <mpi.h>', '#include <mpi.h>\n#include <omp.h>', 1)
src = src.replace('MPI_Init(&argc,&argv);',
                  'int provided; MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);', 1)
# Add pragma above the compute loop (identified by 'for(int i=1;i<=lx;i++) for(int j=1;j<=ly;j++){\n            int k=i*(ly+2)+j;')
if '#pragma omp parallel for' not in src:
    src = src.replace('        for(int i=1;i<=lx;i++) for(int j=1;j<=ly;j++){\n            int k=i*(ly+2)+j;\n            unew[k]',
                      '        #pragma omp parallel for collapse(2) schedule(static)\n        for(int i=1;i<=lx;i++) for(int j=1;j<=ly;j++){\n            int k=i*(ly+2)+j;\n            unew[k]', 1)
(labDir/'heat2Dhyb.c').write_text(src)
sshPut(str(labDir/'heat2Dhyb.c'), env['HPC_LAB_DIR']+'/heat2Dhyb.c')
showFile(labDir/'heat2Dhyb.c', language='c', maxLines=15, title='heat2Dhyb.c (top)')


In [ ]:
checkpoint("Part 1 - hybrid source", [
    check("hyb source has omp pragma",
          fileContains(str(labDir/'heat2Dhyb.c'), '#pragma omp')),
    check("hyb source uses Init_thread",
          fileContains(str(labDir/'heat2Dhyb.c'), 'MPI_Init_thread')),
])


## Part 2 · Ranks-per-node vs threads-per-rank · pick the split

A Crux node has 128 cores. Ways to fill it:

| Ranks/node | Threads/rank | Total cores | Typical fit |
|---|---|---|---|
| 128 | 1 | 128 | pure MPI, most halo data, most ranks |
| 2   | 64 | 128 | one rank per socket, thread within socket |
| 1   | 128 | 128 | one rank per node, all threads, minimum halo |

For memory-bound codes like the heat stencil, **one rank per NUMA domain** (2/node on Crux) is usually the sweet spot. It keeps MPI comm to a minimum while giving each rank a local memory pool.


In [ ]:
# [Hub -> Crux] Sweep three splits at fixed 4 nodes total.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -O3 -fopenmp -o heat2Dhyb heat2Dhyb.c
for cfg in "128 1" "2 64" "1 128"; do
  set -- $cfg
  ppn=$1; tpr=$2; NRANKS=$(( 4 * ppn ))
  echo === $ppn ranks/node, $tpr threads/rank ===
  OMP_NUM_THREADS=$tpr OMP_PROC_BIND=close OMP_PLACES=cores \\
    mpiexec -n $NRANKS --ppn $ppn --depth $tpr --cpu-bind depth \\
    ./heat2Dhyb --N 2048 --steps 200
done
'''
pbsPath = labDir/'hybJob.pbs'
pbsPath.write_text(pbsHeader(name='lab07Hyb', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             select='4:system=crux', walltime='00:20:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/hyb.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/hybJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/hybJob.pbs'); waitJob(jobID, 30, 1800)
sshGet(env['HPC_LAB_DIR']+'/hyb.out', str(labDir/'hyb.out'))
print((labDir/'hyb.out').read_text())


In [ ]:
checkpoint("Part 2 - hybrid split sweep", [
    check("hybrid output", fileExists(str(labDir/'hyb.out'))),
])


## Part 3 · Analyze the sweep

Parse the three configurations out of the log and see which was fastest.


In [ ]:
# [Hub] Extract the three configurations from hyb.out.
import re, pandas as pd
txt = (labDir/'hyb.out').read_text()
rows = []
# each block starts with `=== N ranks/node, M threads/rank ===`
for block in re.split(r'===', txt):
    cfg = re.search(r'(\d+) ranks/node, (\d+) threads/rank', block)
    perf = re.search(r'ranks=(\d+).*wall=([\d.]+)s mlups=([\d.]+)', block)
    if cfg and perf:
        rows.append({'variant':'hybrid',
                     'ppn':int(cfg.group(1)), 'threads':int(cfg.group(2)),
                     'ranks':int(perf.group(1)),
                     'wall_s':float(perf.group(2)), 'mlups':float(perf.group(3))})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
if len(df) > 0:
    best = df.loc[df['mlups'].idxmax()]
    showNote(f'best config: {best["ppn"]} ranks/node x {best["threads"]} threads/rank '
             f'-> {best["mlups"]:.1f} MLUP/s', kind='ok')


In [ ]:
checkpoint("Part 3 - sweep parsed", [
    check("parsed >=1 configs", lambda: (len(df) >= 1, f'{len(df)} configs')),
])


## Part 4 · Why the middle option usually wins

Two extremes:
- **Pure MPI (128/node)**: maximum halo data (every rank borders 4 others; total halo grows with ranks), but zero threading overhead.
- **Pure OMP-in-one-rank (1/node)**: minimum halo (one rank per node), but one rank spans both NUMA sockets → half the threads access remote memory.

**One rank per NUMA domain (2/node)** minimizes both problems: modest halo, all threads within one rank see local memory.


In [ ]:
checkpoint("Part 4 - reasoning captured", [
    check("labEnv persists", fileExists(str(labDir/'labEnv.sh'))),
])


## Part 5 · Roofline reprise

Place the best hybrid MLUP/s on the Crux roofline from lab 02. Where does it sit relative to the memory-bandwidth ceiling? A well-tuned hybrid should be within 2x of the memory roof; if it isn't, revisit first-touch and thread binding.


In [ ]:
# [Hub] Compare best hybrid mlups to memory-bandwidth ceiling.
print('Convert your best mlups to GFLOP/s: mlups * 5 / 1000')
print('Compare against lab 02 roofline peak BW * 0.125 FLOP/byte')
print('If the ratio is > 0.7, congratulations - you are close to the roof.')


In [ ]:
checkpoint("Part 5 - roofline compared", [
    check("hyb output for record", fileExists(str(labDir/'hyb.out'))),
])


## Part 6 · Bridge to lab 08

You've maxed out the CPU story: MPI+OpenMP on many nodes, near the memory bandwidth ceiling. The next question: **can a GPU do this faster?**

Lab 08 keeps the OpenMP source and adds `#pragma omp target` to offload the stencil to an NVIDIA GPU on Polaris. The lab moves from Crux to Polaris (where the GPUs live).


## Wrap up

Moved the spine forward one lab. Ready for the next.


### Lab scorecard


In [ ]:
labSummary("Hybrid MPI + OpenMP")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("Hybrid MPI + OpenMP")
